# 04 — Baselines and evaluation

Train and compare: Random Forest, zero-order ANFIS, first-order ANFIS, SVM.

In [ ]:
import sys
from pathlib import Path
import numpy as np
sys.path.insert(0, str(Path.cwd().parent))

from src.data import load_csv, clean_and_prepare
from src.features import select_features_mi
from src.models import (
    IT2_TSK_ANFIS,
    ANFISZero,
    ANFISFirst,
    train_rf,
    evaluate_rf,
    train_svm,
    evaluate_svm,
    predict_svm,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

path = Path("../data/raw/Data-Melbourne_F_fixed.csv")
if not path.exists():
    path = Path("../../Data-Melbourne_F_fixed.csv")
df = load_csv(path)
df_clean = clean_and_prepare(df)
df_final = select_features_mi(df_clean)
X = df_final.drop(columns=['Energy Consumption']).values
y = df_final['Energy Consumption'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
sc = StandardScaler().fit(X_tr)
X_tr_s = sc.transform(X_tr)
X_val_s = sc.transform(X_val)
X_test_s = sc.transform(X_test)

In [ ]:
rf = train_rf(X_tr_s, y_tr)
print("RF:", evaluate_rf(rf, X_test_s, y_test))

In [ ]:
svm, scaler_X, scaler_y = train_svm(X_tr_s, y_tr, scale=True)
print("SVM:", evaluate_svm(svm, X_test_s, y_test, scaler_X, scaler_y, scale=True))

In [ ]:
anfis0 = ANFISZero(n_clusters=7, X=X_tr_s)
anfis0.fit(X_tr_s, y_tr, X_val_s, y_val, epochs=100, lr=0.01, patience=15)
mse0 = np.mean((anfis0.predict(X_test_s) - y_test)**2)
print("Zero-order ANFIS MSE:", mse0)

In [ ]:
anfis1 = ANFISFirst(n_clusters=7, X=X_tr_s)
anfis1.fit(X_tr_s, y_tr, X_val_s, y_val, epochs=100, lr=0.01, patience=15)
print("First-order ANFIS MSE:", np.mean((anfis1.predict(X_test_s) - y_test)**2))